In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt
import cmocean
import matplotlib as mpl
import scipy.io as sio
import xarray as xr
import gc

def diagnostics_from_LES(
    theta,     # (ntime,nz) LES temperature [°C]
    tw,        # (ntime,nz) LES <w'θ'> in K·m/s
    tau13l,    # (ntime,)   LES τ13l [Pa]
    tau23l,    # (ntime,)   LES τ23l [Pa]
    z,         # (nz,)      depth grid [m], positive downward
    time,
     rho0=1026.95, alpha=-0.2,
    g=9.81, nstar=0.06
):
    """
    Returns
      Hbl : (ntime,) mixed‐layer depth by max |dθ/dz|
      Me  : (ntime,) entrainment PE conversion ∫ w'b' dz
      mstar : (ntime,) mechanical coefficient from Eq.(13)
  
    """
    ntime, nz = tw.shape

    # --- 1) Mixed‐layer depth by max |dθ/dz| (Eq.11 used only for rho if you want) ---
    dTdz = np.gradient(theta, z, axis=1)             # ∂θ/∂z
    idx  = np.argmax(np.abs(dTdz), axis=1)           # level of max |∂θ/∂z|
    Hbl  = z[idx]                                    # positive‐down depth

    # --- 2) Buoyancy flux w'b' (Eq.11 + definition) ---
    #    ρ' = α(θ−θ0),  b' = −(g/ρ0) ρ'  =>  w'b' = −(g/ρ0) α <w'θ'>
    bflux = -(g/rho0) * alpha * tw                # [m²/s³]

    Me    = np.zeros(ntime)
    conv  = np.zeros(ntime)
    mstar = np.zeros(ntime)

    # friction velocity and Stokes magnitude once
    tau_mag = np.hypot(tau13l, tau23l)               # Pa
    ustar   = np.sqrt(tau_mag/rho0)                  # m/s
    
    for t in range(ntime):
        H=Hbl[t]
        if np.isnan(H) or H <= z[0]:
            mstar[t]=np.nan;
            continue
            ###below takes entrainment layer within OSBL
        # --- Eq.(3)+(13): Find entrainment zone [Z_eT,Z_eB] where bflux<0 within OSBL---
        ml = z <= H
        zz = z[ml]
        bf = bflux[t, ml]
        pos_neg = bf<0
        Me[t] = np.trapezoid( bf[pos_neg], zz[pos_neg]) if pos_neg.any() else 0.0     
        # convective term C = ∫₀ᴴ max(0,bf) dz
        pos = bf>0
        conv[t] = np.trapezoid( bf[pos], zz[pos]) if pos.any() else 0.0
        # invert Eq.(13): -Me = m* u*³ + n* conv
        mstar[t] = (-Me[t]-nstar*conv[t]) / (ustar[t]**3)

    return Hbl, Me, mstar, ustar

y_km_list     = [50,30,10,0,-20,-40,-60,-200]
y_folder_list = ['50','30','10','0','S20','S40','S60','S200']
LES_case_list      = ['TC052','TC053','TC054','TC055','TC056','TC057','TC058','TC061']
LES_noLT_case_list = ['TC041','TC042','TC043','TC044','TC045','TC046','TC047','TC050']
les_base = '/archive/bgr/Datasets/LES/Hurr/LES_HUR/'
rho0=1026.95 # reference density [kg/m³]
nstar=0.06  # convective coefficient
alpha=-0.2   # thermal expansion coeff [kg·m⁻³ °C⁻¹]
g=9.81        # gravity [m/s²] 
dt_f=100
dt = dt_f/(24*3600)  # For example, 60 seconds one minutes, shorter time in MOM, 5 minutes
##attention mstar dt time step >= MOM time stepDT
save_dir = "/archive/Qian.Xiao/Qian.Xiao/MOM-examples/SCM_ePBL_imposedmstar_10mps/"
os.makedirs(save_dir,exist_ok=True)
for idx, y_km in enumerate(y_km_list):
    y_tag    = y_folder_list[idx]
    les_case = LES_case_list[idx]
    nlt_case = LES_noLT_case_list[idx]

    # 1) LOAD & IMMEDIATELY EXTRACT LES
    mat = sio.loadmat(os.path.join(les_base, f"{les_case}_PROF.mat"))
    t_les    = mat['t'].squeeze()   / 86400.0
    T_full_les = mat['T'][:] - 273.15  # (ntime, nz)
    tw_les     = -mat['tw'][:]   
    tau13    = mat['tau13l'][:,0]  * rho0
    tau23    = mat['tau23l'][:,0]  * rho0
    z_les    = mat['z'].T.squeeze()
    
    _, _, mstar_les, _=diagnostics_from_LES(
        T_full_les,    # (ntime,nz) LES temperature [°C]
        tw_les,        # (ntime,nz) LES <w'θ'> in K·m/s
        tau13,    # (ntime,)   LES τ13l [Pa]
        tau23,    # (ntime,)   LES τ23l [Pa]
        z_les,         # (nz,)      depth grid [m], positive downward
        t_les,
         rho0=rho0, alpha=alpha,
        g=g, nstar=nstar,
    )    
    time_uniform = np.arange(0, t_les[-1] + dt, dt)
    mstar_interp_LES = np.interp(time_uniform, t_les, mstar_les)
    
    save_path = os.path.join(save_dir, f"LES_results/dt_{dt_f}/time_{y_tag}_LES.txt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
    np.savetxt(save_path, time_uniform, fmt="%.6f")
    save_path = os.path.join(save_dir, f"LES_results/dt_{dt_f}/mstar_{y_tag}_LES.txt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
    np.savetxt(save_path, mstar_interp_LES, fmt="%.6f")
    del mat                          # get rid of everything else   
    gc.collect()

    # 2) LOAD & EXTRACT LES_noLT
    matn = sio.loadmat(os.path.join(les_base, f"{nlt_case}_PROF.mat"))
    t_nlt    = matn['t'].squeeze()   / 86400.0
    tau13n   = matn['tau13l'][:,0] * rho0
    tau23n   = matn['tau23l'][:,0] * rho0
    z_nlt    = matn['z'].T.squeeze()
    T_full_nlt = matn['T'][:] - 273.15  # (ntime, nz)
    tw_nlt    = -matn['tw'][:]       
    _, _, mstar_nlt, _=diagnostics_from_LES(
        T_full_nlt,    # (ntime,nz) LES temperature [°C]
        tw_nlt,        # (ntime,nz) LES <w'θ'> in K·m/s
        tau13n,    # (ntime,)   LES τ13l [Pa]
        tau23n,    # (ntime,)   LES τ23l [Pa]
        z_nlt,         # (nz,)      depth grid [m], positive downward
        t_nlt,
         rho0=rho0, alpha=alpha,
        g=g, nstar=nstar,
    )  
    mstar_interp_LES_noLT = np.interp(time_uniform, t_nlt, mstar_nlt)
    save_path = os.path.join(save_dir, f"LES_noLT_results/dt_{dt_f}/time_{y_tag}_LES_noLT.txt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
    np.savetxt(save_path, time_uniform, fmt="%.6f")
    save_path = os.path.join(save_dir, f"LES_noLT_results/dt_{dt_f}/mstar_{y_tag}_LES_noLT.txt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
    np.savetxt(save_path, mstar_interp_LES_noLT, fmt="%.6f")
    del matn
    gc.collect()